<a href="https://colab.research.google.com/github/rajavi-mhatre/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=hf_token
)

import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{hf_token}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

print(df.columns)

# One row represents one content item (page) for one client on one report date.

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# feature: gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position, report_date
# label: ga4_pageviews
# context: client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, content_hash_id, month
# excluded: client_hash_id (identifier; not predictive), sessions_referral (not relevant to chosen target), scroll_events (not using), ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other (not using in this notebook)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions IS NULL
""")

con.sql(f"""
select
MIN(report_date) as start_date,
MAX(report_date) as end_date
from read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# The data varies, and each client ha different amounts of data available.
# Not every metric is available for each client. For example, GSC data or GA4 data.
# The data can highlight issues for each client, but cannot explain why content performance, be that good or bad.
# Pseudonymized data means that clients can not be identified to resolve issues or provide feedback.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.